# Deutsch-Jozsa Algorithm

The Deutsch-Jozsa algorithm determines whether a function $f:\{0,1\}^n \to \{0,1\}$ is **constant** (same output for all inputs) or **balanced** (outputs 0 for half the inputs and 1 for the other half) using just **1 query**.

Classically, this requires $2^{n-1} + 1$ queries in the worst case. For $n=2$, we use 3 qubits: 2 input qubits and 1 ancilla initialized to $|1\rangle$.

In [ ]:
import cudaq
import numpy as np


@cudaq.kernel
def dj_constant():
    """Deutsch-Jozsa with constant oracle f(x) = 0 (identity)."""
    qubits = cudaq.qvector(3)
    x(qubits[2])
    h(qubits[0])
    h(qubits[1])
    h(qubits[2])
    h(qubits[2])
    h(qubits[0])
    h(qubits[1])


@cudaq.kernel
def dj_balanced_xor():
    """Deutsch-Jozsa with balanced oracle f(x0,x1) = x0 XOR x1."""
    qubits = cudaq.qvector(3)
    x(qubits[2])
    h(qubits[0])
    h(qubits[1])
    h(qubits[2])
    cx(qubits[0], qubits[2])
    cx(qubits[1], qubits[2])
    h(qubits[0])
    h(qubits[1])

In [ ]:
print("=== Constant oracle f(x) = 0 ===")
sv = np.array(cudaq.get_state(dj_constant))
probs = np.abs(sv) ** 2
basis = ["|000>", "|001>", "|010>", "|011>",
         "|100>", "|101>", "|110>", "|111>"]
for b, prob in zip(basis, probs):
    if prob > 0.001:
        print(f"  {b}: P={prob:.4f}")
result = cudaq.sample(dj_constant, shots_count=1000)
for bitstring, count in result.items():
    print(f"  measured |{bitstring}>: {count}")
print("  Result: CONSTANT (all zeros after final H)")

In [ ]:
print("=== Balanced oracle f(x) = x0 XOR x1 ===")
sv2 = np.array(cudaq.get_state(dj_balanced_xor))
probs2 = np.abs(sv2) ** 2
for b, prob in zip(basis, probs2):
    if prob > 0.001:
        print(f"  {b}: P={prob:.4f}")
result2 = cudaq.sample(dj_balanced_xor, shots_count=1000)
for bitstring, count in result2.items():
    print(f"  measured |{bitstring}>: {count}")
print("  Result: BALANCED (at least one input qubit is |1>)")
print("\nDeutsch-Jozsa: constant => |00>, balanced => anything else")